In [1]:
import gymnasium as gym
import numpy as np
import math
import os
import configparser
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt
from sb3_contrib.common.maskable.utils import get_action_masks



In [2]:
from src.hpc_env import HPCenv
from src.validation import Validation
from src.training import Train
from src.baseline import MedianBaseline
from src.utils import mask_fn, get_config_as_dict
from src.carbon_intensity import CarbonIntensity

In [3]:
WORKLOAD_PATH = "data/workloads/lublin_256.swf"

# Load config with explicit path and typed parsing
config = configparser.ConfigParser()
config_path = os.path.join(os.getcwd(), 'config_file', 'config.ini')
config.read(config_path)

GAE_LAMBDA = config.getfloat('training', 'gae_lambda')
GAMMA = config.getfloat('training', 'gamma')
EPISODE_LENGTH = config.getint('training', 'episode_length')

## Model training

In [4]:
config_dict = get_config_as_dict(config) 
print(config_dict)
train = Train(config_dict=config_dict, workload_path=WORKLOAD_PATH, run_id="ratio_reward")

{'use_constant_power': True, 'constant_power_per_processor': 500, 'procs_per_node': 1, 'idle_power': 15, 'carbon_year': 2021, 'variable_carbon_intensities': False, 'carbon_granularity': 'minutely', 'green_forecast_length': 24, 'eta': 0.0, 'max_queue_size': 10, 'run_win_length': 64, 'delay_time_list': [300, 3600], 'delay_time_list_length': 2, 'max_wait_n_jobs': 1, 'job_feature': 5, 'run_feature': 2, 'green_feature_pr_timeslot': 1, 'green_feature_constant': 8, 'episode_length': 256, 'gamma': 0.999, 'gae_lambda': 0.97, 'batch_size': 512, 'seed': 0, 'n_epochs': 5, 'pi_nn': [256, 256], 'vf_nn': [256, 256], 'n_steps': 10240, 'total_timesteps': 50480000, 'ent_coef': 0.05, 'learning_rate': 0.0001, 'base_line_wait_carbon_penality': 0.01, 'bounded_slowdown_threshhold': 10, 'reward_type': 'delay_vs_now_reward', 'max_power': 19000, 'max_green': 19000, 'max_wait_time': 200000, 'max_run_time': 162754, 'max_requested_processors': 256}
Repository created at ./resultsratio_reward/ and config saved to .

In [ ]:
train.run(save_checkpoints=True)

Logging to ./resultsratio_reward/seed_0_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 463      |
|    ep_rew_mean     | 388      |
| time/              |          |
|    fps             | 1329     |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 10240    |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 460           |
|    ep_rew_mean          | 324           |
| time/                   |               |
|    fps                  | 1278          |
|    iterations           | 2             |
|    time_elapsed         | 16            |
|    total_timesteps      | 20480         |
| train/                  |               |
|    approx_kl            | 0.00036446552 |
|    clip_fraction        | 0.0286        |
|    clip_range           | 0.05          |
|    entropy_loss         | -2            |
| 